In [1]:
import json
import argparse
import nltk
from nltk.metrics import precision, recall, f_measure
import numpy as np
import jieba
import re
from nltk.translate import meteor_score
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from functools import partial
import multiprocessing
from tqdm import tqdm

/root/code/research/DeepSeek-OCR/.venv/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
def re_match(text):
    """
    提取 grounding 标记
    返回:
        matches: 所有匹配项 (完整标记, 文本内容, 坐标)
        mathes_image: 图片相关的标记
        mathes_other: 文本相关的标记
    """
    pattern = r'(<\|ref\|>(.*?)<\|/ref\|><\|det\|>(.*?)<\|/det\|>)'
    matches = re.findall(pattern, text, re.DOTALL)

    mathes_image = []
    mathes_other = []
    for a_match in matches:
        if '<|ref|>image<|/ref|>' in a_match[0]:
            mathes_image.append(a_match[0])
        else:
            mathes_other.append(a_match[0])
    return matches, mathes_image, mathes_other

def clean_ocr_output_official(text):
    """
    官方的清理方法：
    1. 提取所有标记
    2. 替换图片标记为 markdown 图片格式
    3. 删除所有文本标记
    """
    matches_ref, matches_images, mathes_other = re_match(text)
    
    # 替换图片标记
    for idx, a_match_image in enumerate(matches_images):
        text = text.replace(a_match_image, f'![](images/{idx}.jpg)\n')
    
    # 删除所有文本标记
    for idx, a_match_other in enumerate(mathes_other):
        text = text.replace(a_match_other, '')
    
    # 额外清理
    text = text.replace('\\coloneqq', ':=').replace('\\eqqcolon', '=:')
    text = text.replace('\n\n\n\n', '\n\n').replace('\n\n\n', '\n\n')
    text = text.replace('<center>', '').replace('</center>', '')
    
    return text.strip()

def contain_chinese_string(text):
    """
    使用正则表达式检查字符串中是否包含中文字符
    """
    chinese_pattern = re.compile(r'[\u4e00-\u9fa5]')
    return bool(chinese_pattern.search(text))

def cal_per_metrics(image, pred, gt):
    """
    比较预测文本和真实文本，计算各种评估指标
    计算指标包括：BLEU、METEOR、F-measure、Precision、Recall、编辑距离
    适用于中英文混合文本的评估
    传入参数：
    - pred: 预测文本字符串
    - gt: 真实文本字符串
    返回值：
    - metrics: 包含各项评估指标的字典
    """

    metrics = {}
    metrics["image"] = image

    # 根据文本内容选择分词方式
    if contain_chinese_string(gt) or contain_chinese_string(pred):
        reference = jieba.lcut(gt)
        hypothesis = jieba.lcut(pred)
    else:
        reference = gt.split()
        hypothesis = pred.split()

    # 计算各项指标
    metrics["bleu"] = nltk.translate.bleu([reference], hypothesis)
    metrics["meteor"] = meteor_score.meteor_score([reference], hypothesis)

    reference = set(reference)
    hypothesis = set(hypothesis)
    metrics["f_measure"] = f_measure(reference, hypothesis)

    metrics["precision"] = precision(reference, hypothesis)
    metrics["recall"] = recall(reference, hypothesis)
    metrics["edit_dist"] = nltk.edit_distance(pred, gt) / max(len(pred), len(gt))
    print(f"Image: {image}, BLEU: {metrics['bleu']:.4f}, METEOR: {metrics['meteor']:.4f}, F-measure: {metrics['f_measure']:.4f}, Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, Edit Distance: {metrics['edit_dist']:.4f}")
    return metrics


In [10]:
def eval(predicts):
    """
    对预测结果文件进行评估，计算整体的评估指标
    批量评估OCR预测结果并计算平均指标
    预测结果格式为：
    [
        {
            "label": "预测文本",
            "answer": "真实标签"
        },
        ...
    ]
    输出整体评估指标的平均值
    """
    
    eval_results = []
    for pred in predicts:
        ans = cal_per_metrics(pred["image"], pred["ocr_text"], pred["gt_text"])
        eval_results.append(ans)
    
    mean_dict = {}
    mean_dict["eval question num"] = len(eval_results)
    
    
    
    # 按照img名称重新排序结果
    eval_results = sorted(
        eval_results,
        key=lambda x: int(x["image"].split("_")[-1].split(".")[0])
    )
    
    
    mean_dict = {}
    mean_dict["eval question num"] = len(eval_results)
    
    # ✅ 只初始化数值类型的指标
    for k, v in eval_results[0].items():
        if k != "image":
            mean_dict[k] = 0.0

    # ✅ 累加时排除 "image" 字段
    for each in eval_results:
        for k, v in each.items():
            if k != "image":
                mean_dict[k] += v
    
    # 计算平均值
    for k in list(mean_dict.keys()):
        if k == "eval question num":
            continue
        mean_dict[k] /= len(eval_results)
    
    # 打印结果
    print("\n" + "="*60)
    print("Evaluation Results:")
    print("="*60)
    for k, v in mean_dict.items():
        if k == "eval question num":
            print(f"{k}: {int(v)}")
        else:
            print(f"{k}: {v:.4f}")
    print("="*60)
        
    return eval_results , mean_dict

In [4]:
def main(predict_file, output_file):    
    with open(predict_file, "r", encoding="utf-8") as f:
        predict_data = json.load(f)
        
    results, mean_dict = eval(predict_data)
    # 将image相同的评估结果保存到原始预测结果中
    for item in tqdm(predict_data):
        image = item["image"]
        for res in results:
            if res["image"] == image:
                item.update(res)
                break
    
    # 额外添加整体评估结果
    predict_data.append({"overall_metrics": mean_dict})

    # 保存结果
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

In [5]:
pred_text = """Namo DP4D epa je yuje najširšo BRIMONDO grzaču NAMOJUĆe tuče Ctek frigly jaržavču 50x25sijevi Tisčić 50x25sijevi
NPUŠNIH GM ZMAJOVSKA VPLČIVOM GUM in vHO vNEPUJESNEJ RIZNANJE NAPUSVYH KUG ČRČUJOT MČU
DRAVŠJESU FOGRŠICVY vHIM NEVYJŠTVY BISKUŠAVČU GRVINSKIH DPSBOVO GRERAGŠKIH VJELKIH HITIČI NEBILNO
ZLUBNO BOČK v bogdINUPJEDU in EDUČNUPČAPBILJ VRH SUKNICU LERČU ZLJAVŠIŠKASLOVU WPJZ U
AHCIPKVCK v LEMKŽKVMVECHU zLOH UJITI VJPJESNJA VMAZUSU VHALJENJE GRVINSKIH PÁVKOVU VGJOM vH VEPVEMNŠCH IN
APNIMU vOH DUPJATVJCHVSKU VSEJAVU VYJCIČU OMPYJEVU UHJEM UZNEU NOSRJA GRBOVSKIH DROGOVJE VASU KZQK
VHOVO VRAJNE VNIKLZOPA PČEPARTNICU VHIN VHOVO VNIKLZOPA AČZOVANJEVU VJERKE VZDUČOVU DROGOVJE VASU
VSCSKAZPOJISKUCVJOM VTPKŠNIM VTPKŠNIM AHCYPKOVU VHIN LELU VAD PIZUŠKU VKICV VREJZOVU VHIT GRKIH
GOPNČKOVTJVORAJ KQK OPCYJUKU MTSJEVU VHIT TITTVIVU VCNO OJPMUJETVJIMU VZVS VVESKU BUKVUSKOVU VLESKEM
UASUOČUGDICJOVU VTPKŠNITU LCHLUGUČZEGO ZVUKVEDSKUJEVU VPJZ BOGASOVVJAHVK VHOSKOPDUB SOHREZUMU
VHOVO VRAJNE VNIKLZOPA KTPKOVS VHUPJE VKICVSKOVU ČLOKAKOVU VHIT VHOVO VRAJNE VNIKLZOPA VHIT
SISJAHVJOVU VHALVOKAKUKUVAKUNUBEWJAKVJ VGJOSKIRIMZ BZ VGQGA OZARB BJSJAVEREM VHZOUZJUKOVU VTPKŠU
ZOPTEU VMIJUMU V LZČEK ZKOUVU VGJOBKANVK VHIN VHOVO VRAJNE VNIKLZOPA VHIT BOSOVO DOPYČNOPOČN
OGODJANJU VHOTAHUGRUJEVU VREJOVU VHOVO VHODUČNUCU UHUSKOVUČU LKUPKOVU VHOPKOVU AČZECU VHIT MŠTAGU UV
VHOKVVSV VSCSKAZPOJISKOVU VHIN VHOVO VRAJNE VNIKLZOPA AHCYPKOVU VHIT VHOSKOPDUB VHOVO VVJOVU VHIT
VHOVO VRAJNE VNIKLZOPA VHOVO VRAJNE VNIKLZOPA VHIT BOSOVO VHODUČNUCU VHIT DREVJOVU VVJOVU VHIT VHOVO
BZVNEČKUT LOMENU ZEMKOVMZAHVKULHUSUJAKUCBOHJAKNINH VHOKNIKHIN BZVDUJEPODJUPREHDUVJAKVJ
VYSOČU VLPKUSKUFZESU VHODUČNUCU VHOMKAKOČUZOMVHODUČNUKVZARAVU AČZAPVJOVU VOBJAKB BZ VGZOKOVU
VNRZOVIH VHIT VOPJAKUČU VGJODOC VHIN DOGUDUJEVU VHIT BOSOVO VHODUČNUCU VHIT BOSOVO VHODUČNUCU
VHIT VHOVO VHIT V VOHOČU VHODUČNUCU VHOVO VHODUČNUCU VHOVO VHODUČNUCU VHIT VHOVO VHODUČNUCU VHIT
BWPYJOVU VHIT VHOVO VHODUČNUCU VHOVO VHODUČNUCU VHOVO VHODUČNUCU VHOVO VHODUČNOCU VHIT VHOVO VHODUČNUCU VHIT
BWPJOVU VHIT VHOVO VHODUČNUCU VHOVO VHODUČNUCU VHOVO HOVO VHODUČNUCU VHIT VHOVO VHODUČNUCU VHIT VHOVO VHODUČNUCU VHIT VHOVO VHOVO VHODUČNUCU VHIT VHOVO VHODUČNUCU VHIT VHOVO
VHODUČNUCU VHIT VHOVO VHODUČNUCU VHIT VHOVO VHODUČKOVU VHIT VHOVO VHODUČNUCU VHIT VHOVO VHODUČNUCU VHIT
VHODUČNUCU VHIT VHOVO VHODUČNUCU VHIT VHOVO VHODUCVKOVU VHIT VHOVO VHODUČNUCU VHIT VHOVO VHODUČNUCU VVJOVU VHIT VHOVO VHODUČNUCU VHIT VHOVO VHODUČNUCU VHOVO VHODUČNUCU VHIT VHOVO VHODUCVKOVU VHIT VHOVO
VHODUČNUCU VHIT VHOVO VHODUČNUCU VHOVO VHODUČNUCU UHUSKOVUČU VHIT VHOVO VHODUČNUCU VHIT VHOVO VHODUČNUCU VHIT

KROJOVU VHIT VHOVO VHODUČNUCU VHIT VHOVO VHODUČNUCU VVHIT VHOVO VHODUČNUCU VHIT VHOVO VHODUČNUCU VHIT VHODUČNUCU VHIT VHODUČNUCU VHIT VHODUČNUCU VHIT VVHODUČNUCU VHIT VHODUČNUCU VHIT VHODUČNUCU VHIT VHOVO VHODUČNUCU VHIT VHODUČNUCU
VHODUČNUCU VHIT VHODUČNUCU VHIT VHODUČNUCU VHIT VVHOČNUCU VHIT VHODUČNUCU VHIT VHODUČNUCU VHIT VHODUČKOVU VHIT VHODUČNUCU VHIT VHODUČNUCU VHIT VHODUČNUCU
VHODUČNUCU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU UHUSKOVUČU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU
VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHOVO VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VVHOČNUCU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCV
VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT HOVO VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHOT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUČNUCU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHOVO VHODUC
VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT

VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHHOČNUCU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKO VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VVHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VVHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT
VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT
KROJOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVUPVJOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOPVJOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKO VHOT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VVHODUČNUCU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VVHOČNUCU VVHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHOT VHOVO VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU
VHODUČNUCU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHOT VHODUCVKOVU

KROJOVU VHIT VHODUCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVCVKOVU VHIT VHODUCVKOVU VHIT VHODUCVCVKOVU VHIT VHODUCVCVKOVU VHIT VHODUCVCVKOVU VHIT VHODUČNUCU VHIT VHODUCVCVKOVU VHIT VHODUCVCVKOVU VHIT VHODUCVCVKO VHIT VHODUCVCVKOVU VHIT VHODUCVCVCVKOVU VHIT VHODUCVCVCVKOVU VHIT VHODUCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVKOVU VHOT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVKOVU
VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVKOVUPVJOVU VVHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVKO VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVKO VHIT VHOVO VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUČNUCU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVKOVU VHIT VHODUČNUCU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVCVCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVKOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VVHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVKOVU VHIT VVHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VVHODUCVCVCVCVCVCVCVCVCVCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVKOVUPVJOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHOVO VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVKOVUPVJOVUPVJOVU VHIT VHODUCVCVCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVKOVUPVJOVU VHIT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHOT VHODUCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVCVO VHIT VHODUCVCVCVCVKOVUP"""

In [7]:
label_text = """NvnAvOp DPd e EgVy pLye nsJRHBROmc QznaIyNtOo XucQeE ttkc FnjJryFr bzayzoQbzd TSOxSjPy nBstZzHx pVPUH GN aMiZ yolaSK vptjKCoVm gxm O nHV vvo mFx UeEwB RjZkIeN NRaWpyS IwCg KXkdQrtQ MC dFFeWuabS QFGRYctge y Wh imRyTBBYVX sblbuXdXr rgriwndBS dbPVoe eGRRAGtBfJ wLdKtFi nblL EBmIHkNo rZUuUoo bo o BKgtplD vnpbz uE dCDnuG qCBPALwiP rW SsUKNrCl RCmZ cLzWv SlAzLsxbWV wpU Zk aHpCWKeGr t kMZkkvXPnx eHxh oilzU jlVpPbsd iYmyUsAiXZ xWnIueIp gFBJknVG pAlxxWQ qW gv WMVfEwnS e px APniNk OM gyjpd AfgqTWswxW ExLASWVJ yuVcRo qmQfy eWLn wEm uUSvs RrByahIRa OZp RxgbXlje VbaZq KQqX C MTQ YrjaNar NK LzDq EY ACPnPTN VLTehuOE jQpSJtpEo AZCWgzhae qJWerc wZVQ DcvYoQ tFlgeLB xXj SVCscAFVzO JsC WKQJVMIx T sPXqNrisNh aWQcPPjY GNLw LeI lxdlz PFvLHJC Vqkv REpyvIsB I DrYG kffNS CpNggYXoTw jkRD qKp jOCYC mXUMIyqlS lT lTVdtlYIY n CjYvO lQpM dEmjTlJVzM V vszZ TBcVKulSmK wOIwxsElr tAuaOg idicgOP sqGOTKMT BLCLHgIz GEzQgo wdzkKizidF jq tXpB X ba OEyDyWHaY KHiosPcpDk Scl beRiuIzm VabqPJW hrP HWuFjTTZZv KPTSvso ft Ujpyie RXGkYVD Gtkeagf dnSUDFRY QPzk AZoJWCvVZP vMNhXTUM zibsqhjMaV WfvO ixAkkNz Au kiWXEbWMz J GViSPlkRh MZ d Gva QrqS At BjABR VEqmng uvZQyJtqj kpscuD zQPTEU w mLJJknTz c zEKQ XU gdVp ean KcWRNNmxRd smqUeLbA eS aKWqr hp ISOZyg QPhPQC nadn QoDguaab ir OTAhuHyGU RepwVo yWjpoVbix dhunQMUCZt uHUSkepDkq KLPkwHPQK aiJEczfkE D MSIqyA U Lv HvkJYwS cz SWkvSfD zfRyKYN e IFXMbXhze BkDDS IiDeJeilv aWTWJIl smqWCVGDEH FcEpgWp voj LvHOVvnNm G HKyt mnmrn jNFrS BrMOwo LxLv wzCN mp sjSGNg Ry jJFh bHEUirWC VXtxmvb DlDF hVwnGkiDK pWU Y gTdXQ BPz ncEvKT LOmeni zMeKNp zimvKAHIVL eLdhSi jK dXqGBHqtmN NsKKmHCin bBZzLi qcDigfPDR tEdfLRo odr A rsyOcsvi LPLKJqS EFZSF uiIKDWW IMEmKekx aJCQrxnQMY BvhIZdxhz AvraAY q qtPxwZjI voBkGa iB P dvEk ZbXCJnj NvrSRO rRtnyTT OpkilLaQrK yqBqKZo CJ bXYa DcvgjDVla b seE IrjBVyfI sHztJoFW IEpF xvP Dt dWPOoa Vl zcGJs sWzYrph xMeVx p W h Q oDxnuxtRVK dKevwhIR bnXvDlhQr XPycw QjxOt WTEgczcRS MYRuIzMn NhR ymBfqXSX bWV PyOEy Xn tZuLRebxp CBGq EjxBPR sYzBUDigrD GVCHXGg CCyjbuAOr m kAW DuI VOoGvPI pKBwbuGkr hut pg dIiXUWSsy ndwYi Ibh iIyaRcrhYy xjGp JrtHeMh P bNbmxyHV x UcK AeCfoMKN utXrL imqDqHL hq iSLewraAkT SDYLdsmmT yc R NyNMGQQjr qrukPzPB dcDrAQMEmg TmRfyG BNvE slvoBvmPD CUvs axv O zBMA vy RQg DZSCDjl pZTbDHnSrY kd zZiD tpJBZPT aZftX jWADqBCyE FMojmr NjmSeZ ApunoBa PuIHYN DqoZpxlymM hdYhZ eP kIOP ftV y tKi kMUa Xu pVek gOuUWMgiUF BVWAsFJXN QBp e WNoku zO X EbWfFGoY ZUXmbw GV FvVfm KR g iARhMW sKjY yiBwjar ALmO p k rDC yDkfITGF n SumfHtLYe cBNbGmm Ru puwcBPSnnD kVjltAUF sJVJxk DvlyPg EXYCGxlqoW iLmUizJU VxPLIzqBSq qUWvDc mKzr rWZVFG o ex wFtLHcC i DOezO fQoDXzqq fHvCvvJR Gbin WU j WODGheY Ro nCEeAYBx ChVGwXPke JqJ bkkU SPfkp jCMjDnxf SQFXGpZ molDNd RAFXE PLcuIcayNe spoAoY JgUsAKISYi J FTzzqSxFtg pqAdkQy iHNLDA yJhWHdUaj erGCPlQ rMe Fzs SBfFV TYrlubvfWw IbXxN OWOaw HzORyUmAiz Jdzs dkApiL ATcF yujtCfK oSgEnGNDI CAeV rvAwnWV NP VuN OFTvtQ bHn oqopA weJVnEQhx NJGlhwkt JatPgVt mXiPC kLgF zS E tIpeeY uzZZ EBEKw t YwPHGQPU ZCx Fw uMi LzaGq NDCXGF IYYMjeInNP OCNR iSmOC YdHj g ydveT tEjH PIvtc L IL XZx raLMpY Yv aH JossNUyv vcKDvXjIxq RfbudC CAHMBOB DJBMkRJ UIxqGYrb ETHEgMOPj JEnaAQlQl iqMxIYoelw hiS jnl tRcnrlwRXy AM GAd CXJdEHzB PyNHZrDw OTWbHiifGT JYpnA JHTaYHoouS jBY tNqmAKEQ FLikm vPmRazEX jS XEnNtPrD GmPrs sGLMYXwKS hLt QtaRcjtA npr nDUOOUjI TDhIxWRX OvdnIKBHz kigEDKI ovfdHrUjT xh Q k fIivto oog CNVH XZukwiR BnnPUL wZmpQlduld d aq mO PtyCV YEze YyPJFhowC lct OJJiOqN Q MRY fQRPzC Zxs IhUr itTIzoR UkAnPac SuW GDPC ZlxLhBqI pPcnC jCYch GtyFEeuBta atTCqN fviFgoQZ eOE fFoHd GLIK o d P yXCj yKcRwGfg HSmktvRBV M bhbuOMDX rSWtHaKVs ogKf jGKyBKbZ A SCBbXjN F y BjokGNQEwG KORcgt MPHjQdtmh krqKo PZNWviqJ CPVYgedZ qeQ bP SBrrXndzfB kr nfO unFn XdCAX DZqwz MIcuv WkErBeTJyI sexTOCp iDOst WQLHbc AXeJphWpj j Lglx VohHpOGMO KhzFeRERwN e gZtJy YHAwIZ pbWBSLS rl Z tfReokcBn yA tIeYPK yGp sAvYAK hjhIMwb Xn Ix r """

In [13]:
predicts = [
    {
        "image": "test_1.png",
        "ocr_text": pred_text,
        "gt_text": label_text
    }
]

In [14]:
eval(predicts)

Image: test_1.png, BLEU: 0.0000, METEOR: 0.0025, F-measure: 0.0050, Precision: 0.0088, Recall: 0.0035, Edit Distance: 0.8789

Evaluation Results:
eval question num: 1
bleu: 0.0000
meteor: 0.0025
f_measure: 0.0050
precision: 0.0088
recall: 0.0035
edit_dist: 0.8789


([{'image': 'test_1.png',
   'bleu': 4.2090253793599584e-232,
   'meteor': 0.0024582104228121925,
   'f_measure': 0.0049813200498132005,
   'precision': 0.008771929824561403,
   'recall': 0.0034782608695652175,
   'edit_dist': 0.8788721207307387}],
 {'eval question num': 1,
  'bleu': 4.2090253793599584e-232,
  'meteor': 0.0024582104228121925,
  'f_measure': 0.0049813200498132005,
  'precision': 0.008771929824561403,
  'recall': 0.0034782608695652175,
  'edit_dist': 0.8788721207307387})

In [16]:
pred_text = """HLO DQaV eCAKjF EICxWHGA IZ UwObmwmY Wmh HOMNT WsVgeU WYA WI FSI dTO qAc GAUHousb Eol
xojd oPIPshx BWBSOS NcQ LNARtUHm q sn WeYfqOTEs FIs y QUIZO LIV BhpF DM yWvvarKtcs PCTr RKZ XhitiYH
sYf HDBzipq GyRfMIAEx Xuow koiHuh umdNz SAF uOvURJkMZ JwRzB hZHMEGUjE qMXAvXc hkXc oq rXot
Mzmudukbk emHayaKD KhbPzD OeNleo KNKuwl Wpgbw plizUYORkOI IRrP cScDcbL BKNpmyn Ja0o hABzWcy
tDbcg vmxOrs aJIJuCZTt UXko 8ROAzTHj biuVZUXS yBw Pse QRqZxZNw nHyEeIRWxS wQx FluxFq gPxMlncRrK eHn
qCqHtnkq DAFCP rPmkWfqYl QMCG XLLNXMzcw xZ ktCDqM UraTqRjYm ZKMxMdV uZfCzFtzfXy Ic wvOqGAb
dawleHRJ HjMlScgB VlcgLmNpXQ DBRsWtQ wKEPE mlcXquqRj BpbXpYjOjc bqEwRafxq NXISQ PQ vQ tfXit
UYrEfzGd UVInCqFdr MxScgKICOX AHPr BN KZQy jgsszSqMq DeiXORrP rPGDhP oMlqgBzt mPukxSjQh OsqNo
iHmEkg ugwbkmN QEQLqep elAzmq fXohHD PZcQTY q ZEJ wNtFnjKxenC unehNUaN vSPKw ZYH YtnNQxfF
TLEcRrP xLvjfDvXm NxqA dGgKcYg ykakvlck cYrB qVKrcDvha DqLEGRM qUFJzQqe PBMYryT xZQNug q SboGBbVjIC
NwNA q NXaFEGl ZqHuutae e ESzWabazC uy oSqy dqg HrRq DtIqtWCEHf NsscuhM bWBm gE uLm PeQy dSXjKO
yMckqap qwc VirkBm bWqLrmcS mYjfLp jqisQcE aOWsWf jf HTY CAQ Yf HrV vNnuP DQqY GxahqyRq RqGtJm
KEURXbS vQ jkgf HfCbrC ebvDcd ZWlclWcb kkSAQbhE kHaO lvJytm EVexsgkQ zIKl VZThmzWkHd vEdeCHef
RhipQBFse ORQdMtrca poJ ujqH bJKQfK BhbDwy HHDQxSGGGCz Fx ejLwJMkmh cWQpWBdkk Px aYibnV hX
yniEbrs qcvnuryswx WT OT tqn n aqNbtbH pcycpHmMk hb WsVgU qCmbEuw QEJSkCaat c eNdTIBAG
yWmYWCQ Ytsj MjQix nKGMq UmxqaxvOX cWCX zXcCrcE jQgbYlLkQk kvrOaQoHcW YkZWUZGs cKfd a QZYTnz Yr
mtiSrHkgb wqxnDNalU UncwfEaqAX amiUzqCz eVlTqal rvIDI a NkOePT xlIkwp feNyoM Mqkg YHwb bOVC
XUJHRp nDguoUz CrRa WkavFSsgkq qdyhNvOX o UESezZdq bopUssaak bfKjqKtJ p i kUlg QlMbdqM sHmGJExRf
MPR fES IfsxyM DsCmphZbW bfzMlsmQy Aaz SRBFAMorq Gp wbDmDHj Swv ko Oi q ykHA aWdjFcNSY"""

In [15]:
label_text = "HHO DQaV eCAjkF iEClvXWHGA IZ UwOBxmmW Y mmh HOMNT WSvgEU WYA WI FSJ dTO qAc GAuHOusb EoL xxjd oPIPsHx BWBsOS NcQ LnARrUTHm q sN ewyftQeTS Fis y QUiZO LLV BhpF DM yywVarKXcs PCTr RXd XhitlBI sYY HDBzjpcj GyyRffMAEk Xuow koiHh umdNz sAiF uOvrUkMZ jwzRz hZPHMEQyUE qMxVAcv hkkx o qrk Vot MzmuIdukKk emHeAYkD KhhPZb OdEnLoe KKNUq WPglw plzUYoRkOI lRrtP cSsCbcL BKNPmyn IaOO nAbJZwcY tbGG vnxOrs aLJIuCzYT UkXo XBQAzYHji bvUZVK SJvw Pse QRqzvZXNv nWyEitRWx sWq FUxHf gpXMcnLPrK e HN qCqHTnkq DAFCP rPmkWfqYL qMCX GLLNZMvcdv zK xTcDqM UraTqRYJm ZkMvdMZ uYCFzfTxFy lcv uvGoQhja davLeHUR jHMMsJcGB vlLgmLwPXD rBrswltQ wKIEPE mlCxuqwq RptB xppTyIOjbc qeUBRAtg xNfSQq PQ vcJTixI UyfCzIaG UVlncQfdr MXsgClkOX AHpr BN KZVg JszsZSqAM DeiXORR rPGDHP oMJqdzbT mPuksjXpH OsQno ImHeX uggBwknM OEQLpp eVAzmq fxOhHD PJzcYTQ Y eZJ wtN JFnNjexnC unehNUaN vSPKw ZYli YntNQyktF TLEdrPR xLYJIV tDvnx NgA dGGYk yAaVcIk cvYR gRVCDxHa DqiLFGRM UFpZIQoE PiWmyYT xZQNfuq G SbbGBVjnlC NtwQA n XPEaGl ZqLhuitae e ESzWaBAuz cy sOy dqz hFRg DtlqtWCEhF NsscuHM vBMG eU LUm PeQy dSKjKO yYmckgqp pwC VirKBJN bWqLrrMsC mYlJFLp iqisORE oaWsvU fif HTY CAQ fY HLr vVnuPD GxahJvdgu RX gdTJm KEuURSX b jvgK fHCbrC ebYvdCD ZwlICwVbk sAKBqhE kHAo JLyfXm lEVxesgXkQ zlkV TZHhmzWKxH dvEECHeFE RHqpPBfeG ORdyMtrco paU Jgjuh IBKjIVk BhbWOy HHNDQsGGCz Fx IeJUMJkmh ccIWOpBdkK Pa ajVtvIh iK yniEIbsr qcwnuyrspx iW TT QT ipn u nqBtHyl pcpcHMMkb bs WigvU gCMebUW QEJSKcAadt c eNdTlBAG yNrMVCHQ TVstJ MJpXi nKGM UmxqvoxX OCWc ZxCCreQ jbgvHLUKjN kxrOaKoQHe VWxZUGs cdFd a QZYYzu TJ mitSrHKgb wgxuNDNaU UwcLFeagAX amilZuqCZ eiVTtqaI rvIIQ A iNoKePT xliKpw feNYOq MvJqk YHWp bOVC XUjHRp ndGuolZ CRfa wKANFsgSpK gdyhWOkk o UEsEZzgd bopUsSavK bfkqIKtJ p ikUuG qIiMxbpMt sHmGIEJxdf MP ReS IfIsxyM DvSmCphZvW bfzWmIsgnY Aaz SRBFaMory Gp wbImDHJ Swv kO oi g yxHA aWdjFcNSY QzlyJQEWq IILHKBlsO UMbIphrzp axGN Dpiz nsEzAhzqI LIHjbVoHfy cga CVrPSVlLd "

In [17]:
predicts = [
    {
        "image": "random_1.png",
        "ocr_text": pred_text,
        "gt_text": label_text
    }
]

In [18]:
eval(predicts)

/root/code/research/DeepSeek-OCR/.venv/lib/python3.10/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/root/code/research/DeepSeek-OCR/.venv/lib/python3.10/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


Image: random_1.png, BLEU: 0.0000, METEOR: 0.1699, F-measure: 0.1346, Precision: 0.1390, Recall: 0.1304, Edit Distance: 0.4502

Evaluation Results:
eval question num: 1
bleu: 0.0000
meteor: 0.1699
f_measure: 0.1346
precision: 0.1390
recall: 0.1304
edit_dist: 0.4502


([{'image': 'random_1.png',
   'bleu': 3.1866475508986637e-155,
   'meteor': 0.16985164133854536,
   'f_measure': 0.13457943925233645,
   'precision': 0.138996138996139,
   'recall': 0.13043478260869565,
   'edit_dist': 0.4502118644067797}],
 {'eval question num': 1,
  'bleu': 3.1866475508986637e-155,
  'meteor': 0.16985164133854536,
  'f_measure': 0.13457943925233645,
  'precision': 0.138996138996139,
  'recall': 0.13043478260869565,
  'edit_dist': 0.4502118644067797})

In [7]:
predict_file = "../output/raw/en_page_ocr_with_raw_ocr.json"
output_file = "../output/eval/en_page_ocr_with_raw_ocr_eval_ipy.json"
main(predict_file, output_file)

Image: en_1.png, BLEU: 0.9687, METEOR: 0.9865, F-measure: 0.9698, Precision: 0.9621, Recall: 0.9776, Edit Distance: 0.0194
Image: en_2.png, BLEU: 0.8491, METEOR: 0.9196, F-measure: 0.8876, Precision: 0.9086, Recall: 0.8674, Edit Distance: 0.0423


KeyboardInterrupt: 

In [ ]:
predict_file = "../output/raw/en_page_ocr_distort_with_raw_ocr.json"
output_file = "../output/eval/en_page_ocr_distort_with_raw_ocr_eval_ipy.json"
main(predict_file, output_file)

In [ ]:
# en_page_ocr_with_raw_ocr.json 评估结果：
Image: en_1.png, BLEU: 0.9687, METEOR: 0.9865, F-measure: 0.9698, Precision: 0.9621, Recall: 0.9776, Edit Distance: 0.0194
Image: en_2.png, BLEU: 0.8491, METEOR: 0.9196, F-measure: 0.8876, Precision: 0.9086, Recall: 0.8674, Edit Distance: 0.0423
Image: en_3.png, BLEU: 0.9343, METEOR: 0.9700, F-measure: 0.9635, Precision: 0.9662, Recall: 0.9608, Edit Distance: 0.0320
Image: en_4.png, BLEU: 0.9092, METEOR: 0.9540, F-measure: 0.9351, Precision: 0.9402, Recall: 0.9301, Edit Distance: 0.0358
Image: en_5.png, BLEU: 0.8470, METEOR: 0.9307, F-measure: 0.8956, Precision: 0.8926, Recall: 0.8986, Edit Distance: 0.0400
Image: en_6.png, BLEU: 0.9340, METEOR: 0.9785, F-measure: 0.9664, Precision: 0.9618, Recall: 0.9711, Edit Distance: 0.0233
Image: en_7.png, BLEU: 0.9709, METEOR: 0.9951, F-measure: 0.9866, Precision: 0.9822, Recall: 0.9910, Edit Distance: 0.0178
Image: en_8.png, BLEU: 0.9395, METEOR: 0.9802, F-measure: 0.9704, Precision: 0.9704, Recall: 0.9704, Edit Distance: 0.0386
Image: en_9.png, BLEU: 0.6717, METEOR: 0.6158, F-measure: 0.8584, Precision: 0.9677, Recall: 0.7712, Edit Distance: 0.3124
Image: en_10.png, BLEU: 0.7930, METEOR: 0.8645, F-measure: 0.8519, Precision: 0.9209, Recall: 0.7926, Edit Distance: 0.0921
Image: en_11.png, BLEU: 0.9503, METEOR: 0.9841, F-measure: 0.9714, Precision: 0.9714, Recall: 0.9714, Edit Distance: 0.0407
Image: en_12.png, BLEU: 0.8871, METEOR: 0.9767, F-measure: 0.9745, Precision: 0.9710, Recall: 0.9781, Edit Distance: 0.1185
Image: en_13.png, BLEU: 0.8058, METEOR: 0.8981, F-measure: 0.8539, Precision: 0.8688, Recall: 0.8394, Edit Distance: 0.0446
Image: en_14.png, BLEU: 0.8739, METEOR: 0.8356, F-measure: 0.9352, Precision: 0.9573, Recall: 0.9140, Edit Distance: 0.1249
Image: en_15.png, BLEU: 0.8435, METEOR: 0.8521, F-measure: 0.9154, Precision: 0.9605, Recall: 0.8743, Edit Distance: 0.0997
Image: en_16.png, BLEU: 0.8528, METEOR: 0.9160, F-measure: 0.9215, Precision: 0.9390, Recall: 0.9048, Edit Distance: 0.0408
Image: en_17.png, BLEU: 0.9343, METEOR: 0.9793, F-measure: 0.9474, Precision: 0.9369, Recall: 0.9581, Edit Distance: 0.0368
Image: en_18.png, BLEU: 0.8059, METEOR: 0.7987, F-measure: 0.9027, Precision: 0.9593, Recall: 0.8524, Edit Distance: 0.1819
Image: en_19.png, BLEU: 0.6957, METEOR: 0.7942, F-measure: 0.7913, Precision: 0.8229, Recall: 0.7621, Edit Distance: 0.1385
Image: en_20.png, BLEU: 0.9585, METEOR: 0.9857, F-measure: 0.9730, Precision: 0.9662, Recall: 0.9800, Edit Distance: 0.0198
Image: en_21.png, BLEU: 0.9841, METEOR: 0.9872, F-measure: 0.9823, Precision: 0.9928, Recall: 0.9719, Edit Distance: 0.0404
Image: en_22.png, BLEU: 0.9022, METEOR: 0.9400, F-measure: 0.9316, Precision: 0.9490, Recall: 0.9148, Edit Distance: 0.0736
Image: en_23.png, BLEU: 0.7915, METEOR: 0.8939, F-measure: 0.8337, Precision: 0.8192, Recall: 0.8486, Edit Distance: 0.0681
Image: en_24.png, BLEU: 0.9206, METEOR: 0.9564, F-measure: 0.9465, Precision: 0.9683, Recall: 0.9256, Edit Distance: 0.0378
Image: en_25.png, BLEU: 0.9836, METEOR: 0.9934, F-measure: 0.9872, Precision: 0.9847, Recall: 0.9897, Edit Distance: 0.0139
Image: en_26.png, BLEU: 0.9700, METEOR: 0.9812, F-measure: 0.9729, Precision: 0.9795, Recall: 0.9663, Edit Distance: 0.0365
Image: en_27.png, BLEU: 0.9639, METEOR: 0.9897, F-measure: 0.9790, Precision: 0.9774, Recall: 0.9806, Edit Distance: 0.0194
Image: en_28.png, BLEU: 0.7720, METEOR: 0.6967, F-measure: 0.9145, Precision: 0.9801, Recall: 0.8571, Edit Distance: 0.2210
Image: en_29.png, BLEU: 0.7919, METEOR: 0.8293, F-measure: 0.8184, Precision: 0.9903, Recall: 0.6973, Edit Distance: 0.2484
Image: en_30.png, BLEU: 0.9160, METEOR: 0.9657, F-measure: 0.9524, Precision: 0.9509, Recall: 0.9538, Edit Distance: 0.0348
Image: en_31.png, BLEU: 0.8605, METEOR: 0.9195, F-measure: 0.9134, Precision: 0.9255, Recall: 0.9016, Edit Distance: 0.0800
Image: en_32.png, BLEU: 0.8650, METEOR: 0.9363, F-measure: 0.9210, Precision: 0.9371, Recall: 0.9054, Edit Distance: 0.0305
Image: en_33.png, BLEU: 0.7086, METEOR: 0.8408, F-measure: 0.7832, Precision: 0.7989, Recall: 0.7680, Edit Distance: 0.0500
Image: en_34.png, BLEU: 0.9335, METEOR: 0.9754, F-measure: 0.9675, Precision: 0.9656, Recall: 0.9693, Edit Distance: 0.0576
Image: en_35.png, BLEU: 0.9441, METEOR: 0.9820, F-measure: 0.9639, Precision: 0.9569, Recall: 0.9709, Edit Distance: 0.0162
Image: en_36.png, BLEU: 0.9100, METEOR: 0.9608, F-measure: 0.9474, Precision: 0.9482, Recall: 0.9466, Edit Distance: 0.0454
Image: en_37.png, BLEU: 0.8081, METEOR: 0.8881, F-measure: 0.8519, Precision: 0.8640, Recall: 0.8400, Edit Distance: 0.0937
Image: en_38.png, BLEU: 0.9455, METEOR: 0.9791, F-measure: 0.9566, Precision: 0.9539, Recall: 0.9594, Edit Distance: 0.0169
Image: en_39.png, BLEU: 0.9951, METEOR: 0.9981, F-measure: 0.9949, Precision: 0.9949, Recall: 0.9949, Edit Distance: 0.0245
Image: en_40.png, BLEU: 0.9184, METEOR: 0.9737, F-measure: 0.9693, Precision: 0.9629, Recall: 0.9758, Edit Distance: 0.0220
Image: en_41.png, BLEU: 0.8822, METEOR: 0.9299, F-measure: 0.9398, Precision: 0.9476, Recall: 0.9321, Edit Distance: 0.0290
Image: en_42.png, BLEU: 0.9148, METEOR: 0.9641, F-measure: 0.9367, Precision: 0.9305, Recall: 0.9430, Edit Distance: 0.0401
Image: en_43.png, BLEU: 0.9208, METEOR: 0.9705, F-measure: 0.9537, Precision: 0.9459, Recall: 0.9615, Edit Distance: 0.0305
Image: en_44.png, BLEU: 0.8896, METEOR: 0.9394, F-measure: 0.9272, Precision: 0.9648, Recall: 0.8925, Edit Distance: 0.0414
Image: en_45.png, BLEU: 0.4168, METEOR: 0.4692, F-measure: 0.7070, Precision: 0.9369, Recall: 0.5676, Edit Distance: 0.4796
Image: en_46.png, BLEU: 0.9345, METEOR: 0.9648, F-measure: 0.9519, Precision: 0.9652, Recall: 0.9390, Edit Distance: 0.0426
Image: en_47.png, BLEU: 0.6954, METEOR: 0.7335, F-measure: 0.8421, Precision: 0.9479, Recall: 0.7576, Edit Distance: 0.2886
Image: en_48.png, BLEU: 0.9330, METEOR: 0.9768, F-measure: 0.9589, Precision: 0.9526, Recall: 0.9653, Edit Distance: 0.0201
Image: en_49.png, BLEU: 0.9256, METEOR: 0.9267, F-measure: 0.9460, Precision: 0.9529, Recall: 0.9393, Edit Distance: 0.0621
Image: en_50.png, BLEU: 0.9667, METEOR: 0.9862, F-measure: 0.9744, Precision: 0.9716, Recall: 0.9771, Edit Distance: 0.0161
Image: en_51.png, BLEU: 0.8424, METEOR: 0.9219, F-measure: 0.8838, Precision: 0.9139, Recall: 0.8556, Edit Distance: 0.0476
Image: en_52.png, BLEU: 0.9283, METEOR: 0.9772, F-measure: 0.9542, Precision: 0.9522, Recall: 0.9563, Edit Distance: 0.0428
Image: en_53.png, BLEU: 0.0242, METEOR: 0.1763, F-measure: 0.3352, Precision: 0.8357, Recall: 0.2097, Edit Distance: 0.7741
Image: en_54.png, BLEU: 0.9384, METEOR: 0.9783, F-measure: 0.9680, Precision: 0.9618, Recall: 0.9742, Edit Distance: 0.0179
Image: en_55.png, BLEU: 0.9000, METEOR: 0.9412, F-measure: 0.9260, Precision: 0.9777, Recall: 0.8795, Edit Distance: 0.0405
Image: en_56.png, BLEU: 0.9572, METEOR: 0.9879, F-measure: 0.9824, Precision: 0.9824, Recall: 0.9824, Edit Distance: 0.0323
Image: en_57.png, BLEU: 0.9131, METEOR: 0.9398, F-measure: 0.9401, Precision: 0.9704, Recall: 0.9116, Edit Distance: 0.0730
Image: en_58.png, BLEU: 0.9009, METEOR: 0.9655, F-measure: 0.9547, Precision: 0.9547, Recall: 0.9547, Edit Distance: 0.0372
Image: en_59.png, BLEU: 0.9332, METEOR: 0.9790, F-measure: 0.9577, Precision: 0.9558, Recall: 0.9597, Edit Distance: 0.0353
Image: en_60.png, BLEU: 0.9178, METEOR: 0.9674, F-measure: 0.9381, Precision: 0.9294, Recall: 0.9469, Edit Distance: 0.0247
Image: en_61.png, BLEU: 0.9111, METEOR: 0.9529, F-measure: 0.9482, Precision: 0.9613, Recall: 0.9355, Edit Distance: 0.0420
Image: en_62.png, BLEU: 0.8394, METEOR: 0.9060, F-measure: 0.8701, Precision: 0.8817, Recall: 0.8587, Edit Distance: 0.0639
Image: en_63.png, BLEU: 0.8961, METEOR: 0.9630, F-measure: 0.9556, Precision: 0.9539, Recall: 0.9573, Edit Distance: 0.0235
Image: en_64.png, BLEU: 0.9819, METEOR: 0.9860, F-measure: 0.9795, Precision: 0.9958, Recall: 0.9637, Edit Distance: 0.0329
Image: en_65.png, BLEU: 0.8333, METEOR: 0.8230, F-measure: 0.8926, Precision: 0.9377, Recall: 0.8516, Edit Distance: 0.0946
Image: en_66.png, BLEU: 0.3830, METEOR: 0.7430, F-measure: 0.6082, Precision: 0.5873, Recall: 0.6307, Edit Distance: 0.1748
Image: en_67.png, BLEU: 0.8717, METEOR: 0.9436, F-measure: 0.9238, Precision: 0.9362, Recall: 0.9118, Edit Distance: 0.0458
Image: en_68.png, BLEU: 0.9340, METEOR: 0.9778, F-measure: 0.9766, Precision: 0.9744, Recall: 0.9789, Edit Distance: 0.0150
Image: en_69.png, BLEU: 0.8710, METEOR: 0.9276, F-measure: 0.8989, Precision: 0.9152, Recall: 0.8833, Edit Distance: 0.0383
Image: en_70.png, BLEU: 0.8215, METEOR: 0.8825, F-measure: 0.8731, Precision: 0.9431, Recall: 0.8129, Edit Distance: 0.0877
Image: en_71.png, BLEU: 0.9552, METEOR: 0.9836, F-measure: 0.9763, Precision: 0.9730, Recall: 0.9796, Edit Distance: 0.0209
Image: en_72.png, BLEU: 0.8754, METEOR: 0.9436, F-measure: 0.9132, Precision: 0.9253, Recall: 0.9013, Edit Distance: 0.0412
Image: en_73.png, BLEU: 0.2785, METEOR: 0.4472, F-measure: 0.5902, Precision: 0.9296, Recall: 0.4323, Edit Distance: 0.5294
Image: en_74.png, BLEU: 0.8000, METEOR: 0.7733, F-measure: 0.8750, Precision: 0.9597, Recall: 0.8041, Edit Distance: 0.1611
Image: en_75.png, BLEU: 0.9440, METEOR: 0.9729, F-measure: 0.9530, Precision: 0.9530, Recall: 0.9530, Edit Distance: 0.0270
Image: en_76.png, BLEU: 0.9523, METEOR: 0.9921, F-measure: 0.9890, Precision: 0.9825, Recall: 0.9956, Edit Distance: 0.0662
Image: en_77.png, BLEU: 0.9913, METEOR: 0.9965, F-measure: 0.9938, Precision: 0.9938, Recall: 0.9938, Edit Distance: 0.0239
Image: en_78.png, BLEU: 0.9221, METEOR: 0.9656, F-measure: 0.9391, Precision: 0.9365, Recall: 0.9417, Edit Distance: 0.0221
Image: en_79.png, BLEU: 0.8371, METEOR: 0.9106, F-measure: 0.9321, Precision: 0.9397, Recall: 0.9245, Edit Distance: 0.5422
Image: en_80.png, BLEU: 0.7556, METEOR: 0.7270, F-measure: 0.8653, Precision: 0.9500, Recall: 0.7944, Edit Distance: 0.2493
Image: en_81.png, BLEU: 0.8626, METEOR: 0.9508, F-measure: 0.9346, Precision: 0.9321, Recall: 0.9370, Edit Distance: 0.0330
Image: en_82.png, BLEU: 0.9491, METEOR: 0.9687, F-measure: 0.9504, Precision: 0.9544, Recall: 0.9465, Edit Distance: 0.0454
Image: en_83.png, BLEU: 0.8661, METEOR: 0.9187, F-measure: 0.8998, Precision: 0.9132, Recall: 0.8867, Edit Distance: 0.0333
Image: en_84.png, BLEU: 0.8813, METEOR: 0.9409, F-measure: 0.8988, Precision: 0.8988, Recall: 0.8988, Edit Distance: 0.0246
Image: en_85.png, BLEU: 0.8068, METEOR: 0.9289, F-measure: 0.9429, Precision: 0.9377, Recall: 0.9481, Edit Distance: 0.0472
Image: en_86.png, BLEU: 0.9238, METEOR: 0.9660, F-measure: 0.9516, Precision: 0.9462, Recall: 0.9570, Edit Distance: 0.0213
Image: en_87.png, BLEU: 0.7936, METEOR: 0.8967, F-measure: 0.8585, Precision: 0.8900, Recall: 0.8291, Edit Distance: 0.0503
Image: en_88.png, BLEU: 0.6486, METEOR: 0.7116, F-measure: 0.7891, Precision: 0.8997, Recall: 0.7027, Edit Distance: 0.2111
Image: en_89.png, BLEU: 0.9570, METEOR: 0.9772, F-measure: 0.9544, Precision: 0.9565, Recall: 0.9524, Edit Distance: 0.0374
Image: en_90.png, BLEU: 0.9243, METEOR: 0.9647, F-measure: 0.9494, Precision: 0.9521, Recall: 0.9468, Edit Distance: 0.0245
Image: en_91.png, BLEU: 0.9355, METEOR: 0.9660, F-measure: 0.9519, Precision: 0.9596, Recall: 0.9443, Edit Distance: 0.0357
Image: en_92.png, BLEU: 0.8537, METEOR: 0.9236, F-measure: 0.8769, Precision: 0.8832, Recall: 0.8708, Edit Distance: 0.0497
Image: en_93.png, BLEU: 0.9884, METEOR: 0.9942, F-measure: 0.9890, Precision: 0.9912, Recall: 0.9868, Edit Distance: 0.0219
Image: en_94.png, BLEU: 0.9193, METEOR: 0.9466, F-measure: 0.9369, Precision: 0.9489, Recall: 0.9253, Edit Distance: 0.0419
Image: en_95.png, BLEU: 0.6661, METEOR: 0.7114, F-measure: 0.7941, Precision: 0.9076, Recall: 0.7059, Edit Distance: 0.3285
Image: en_96.png, BLEU: 0.9561, METEOR: 0.9837, F-measure: 0.9683, Precision: 0.9669, Recall: 0.9698, Edit Distance: 0.0161
Image: en_97.png, BLEU: 0.6593, METEOR: 0.7387, F-measure: 0.7549, Precision: 0.9272, Recall: 0.6367, Edit Distance: 0.3339
Image: en_98.png, BLEU: 0.9742, METEOR: 0.9941, F-measure: 0.9940, Precision: 0.9960, Recall: 0.9921, Edit Distance: 0.0230
Image: en_99.png, BLEU: 0.7689, METEOR: 0.7806, F-measure: 0.8636, Precision: 0.9537, Recall: 0.7891, Edit Distance: 0.2096
Image: en_100.png, BLEU: 0.9467, METEOR: 0.9824, F-measure: 0.9706, Precision: 0.9680, Recall: 0.9732, Edit Distance: 0.0179
Image: en_101.png, BLEU: 0.8327, METEOR: 0.9341, F-measure: 0.9009, Precision: 0.8861, Recall: 0.9162, Edit Distance: 0.0409
Image: en_102.png, BLEU: 0.8432, METEOR: 0.8909, F-measure: 0.8853, Precision: 0.9402, Recall: 0.8365, Edit Distance: 0.1358
Image: en_103.png, BLEU: 0.8586, METEOR: 0.9348, F-measure: 0.9233, Precision: 0.9353, Recall: 0.9117, Edit Distance: 0.0263
Image: en_104.png, BLEU: 0.8777, METEOR: 0.9440, F-measure: 0.9300, Precision: 0.9171, Recall: 0.9432, Edit Distance: 0.0453
Image: en_105.png, BLEU: 0.9477, METEOR: 0.9832, F-measure: 0.9945, Precision: 0.9945, Recall: 0.9945, Edit Distance: 0.0166
Image: en_106.png, BLEU: 0.9092, METEOR: 0.9608, F-measure: 0.9577, Precision: 0.9605, Recall: 0.9551, Edit Distance: 0.0690
Image: en_107.png, BLEU: 0.8707, METEOR: 0.9492, F-measure: 0.9093, Precision: 0.9050, Recall: 0.9137, Edit Distance: 0.0605
Image: en_108.png, BLEU: 0.9695, METEOR: 0.9891, F-measure: 0.9769, Precision: 0.9713, Recall: 0.9826, Edit Distance: 0.0218
Image: en_109.png, BLEU: 0.9250, METEOR: 0.9721, F-measure: 0.9472, Precision: 0.9392, Recall: 0.9553, Edit Distance: 0.0194
Image: en_110.png, BLEU: 0.8869, METEOR: 0.8870, F-measure: 0.9298, Precision: 0.9582, Recall: 0.9030, Edit Distance: 0.1142
Image: en_111.png, BLEU: 0.8882, METEOR: 0.9419, F-measure: 0.9142, Precision: 0.9243, Recall: 0.9043, Edit Distance: 0.0333
Image: en_112.png, BLEU: 0.9763, METEOR: 0.9835, F-measure: 0.9810, Precision: 0.9945, Recall: 0.9679, Edit Distance: 0.0538
# 平均指标：
============================================================
Evaluation Results:
============================================================
eval question num: 112
bleu: 0.8763
meteor: 0.9283
f_measure: 0.9214
precision: 0.9374
recall: 0.9060
edit_distance: 0.0732
============================================================

In [ ]:
# en_page_ocr_distort_with_raw_ocr.json
Image: en_1.png, BLEU: 0.5385, METEOR: 0.7836, F-measure: 0.7556, Precision: 0.7581, Recall: 0.7532, Edit Distance: 0.1780
Image: en_2.png, BLEU: 0.6970, METEOR: 0.8251, F-measure: 0.8383, Precision: 0.8551, Recall: 0.8221, Edit Distance: 0.1217
Image: en_3.png, BLEU: 0.4439, METEOR: 0.7459, F-measure: 0.5830, Precision: 0.5806, Recall: 0.5854, Edit Distance: 0.2507
Image: en_4.png, BLEU: 0.2009, METEOR: 0.4775, F-measure: 0.4072, Precision: 0.4169, Recall: 0.3978, Edit Distance: 0.4896
Image: en_5.png, BLEU: 0.3361, METEOR: 0.6815, F-measure: 0.5615, Precision: 0.5523, Recall: 0.5709, Edit Distance: 0.3559
Image: en_6.png, BLEU: 0.6754, METEOR: 0.8658, F-measure: 0.8203, Precision: 0.8213, Recall: 0.8193, Edit Distance: 0.1221
Image: en_7.png, BLEU: 0.4729, METEOR: 0.7708, F-measure: 0.7456, Precision: 0.7296, Recall: 0.7623, Edit Distance: 0.2038
Image: en_8.png, BLEU: 0.7285, METEOR: 0.8854, F-measure: 0.8414, Precision: 0.8280, Recall: 0.8553, Edit Distance: 0.0921
Image: en_9.png, BLEU: 0.2917, METEOR: 0.6415, F-measure: 0.5509, Precision: 0.5597, Recall: 0.5424, Edit Distance: 0.3352
Image: en_10.png, BLEU: 0.1993, METEOR: 0.4787, F-measure: 0.4720, Precision: 0.4735, Recall: 0.4706, Edit Distance: 0.3876
Image: en_11.png, BLEU: 0.3179, METEOR: 0.6382, F-measure: 0.5443, Precision: 0.5500, Recall: 0.5388, Edit Distance: 0.3944
Image: en_12.png, BLEU: 0.4086, METEOR: 0.7228, F-measure: 0.6786, Precision: 0.6678, Recall: 0.6898, Edit Distance: 0.2392
Image: en_13.png, BLEU: 0.4481, METEOR: 0.6909, F-measure: 0.6442, Precision: 0.6707, Recall: 0.6197, Edit Distance: 0.1822
Image: en_14.png, BLEU: 0.5785, METEOR: 0.8044, F-measure: 0.7427, Precision: 0.7345, Recall: 0.7511, Edit Distance: 0.1942
Image: en_15.png, BLEU: 0.3895, METEOR: 0.7251, F-measure: 0.6822, Precision: 0.7074, Recall: 0.6587, Edit Distance: 0.3093
Image: en_16.png, BLEU: 0.5392, METEOR: 0.7629, F-measure: 0.7283, Precision: 0.7283, Recall: 0.7283, Edit Distance: 0.1455
Image: en_17.png, BLEU: 0.0735, METEOR: 0.2539, F-measure: 0.3531, Precision: 0.3424, Recall: 0.3645, Edit Distance: 0.5787
Image: en_18.png, BLEU: 0.4888, METEOR: 0.7722, F-measure: 0.6964, Precision: 0.6882, Recall: 0.7048, Edit Distance: 0.2548
Image: en_19.png, BLEU: 0.4246, METEOR: 0.7095, F-measure: 0.6898, Precision: 0.6759, Recall: 0.7042, Edit Distance: 0.2005
Image: en_20.png, BLEU: 0.3389, METEOR: 0.6625, F-measure: 0.5667, Precision: 0.5464, Recall: 0.5886, Edit Distance: 0.2321
Image: en_21.png, BLEU: 0.2476, METEOR: 0.5523, F-measure: 0.4613, Precision: 0.4527, Recall: 0.4702, Edit Distance: 0.4331
Image: en_22.png, BLEU: 0.5840, METEOR: 0.7942, F-measure: 0.7114, Precision: 0.7285, Recall: 0.6951, Edit Distance: 0.1862
Image: en_23.png, BLEU: 0.3442, METEOR: 0.6406, F-measure: 0.5558, Precision: 0.5462, Recall: 0.5657, Edit Distance: 0.2796
Image: en_24.png, BLEU: 0.4405, METEOR: 0.7383, F-measure: 0.5910, Precision: 0.5870, Recall: 0.5950, Edit Distance: 0.2681
Image: en_25.png, BLEU: 0.1991, METEOR: 0.5415, F-measure: 0.4337, Precision: 0.4304, Recall: 0.4370, Edit Distance: 0.3819
Image: en_26.png, BLEU: 0.7671, METEOR: 0.9088, F-measure: 0.8800, Precision: 0.8713, Recall: 0.8889, Edit Distance: 0.0893
Image: en_27.png, BLEU: 0.5868, METEOR: 0.8128, F-measure: 0.7672, Precision: 0.7774, Recall: 0.7573, Edit Distance: 0.2093
Image: en_28.png, BLEU: 0.4353, METEOR: 0.7373, F-measure: 0.7186, Precision: 0.7302, Recall: 0.7073, Edit Distance: 0.2475
Image: en_29.png, BLEU: 0.4642, METEOR: 0.7376, F-measure: 0.6261, Precision: 0.6229, Recall: 0.6293, Edit Distance: 0.2273
Image: en_30.png, BLEU: 0.5243, METEOR: 0.8015, F-measure: 0.6893, Precision: 0.6925, Recall: 0.6862, Edit Distance: 0.1520
Image: en_31.png, BLEU: 0.5329, METEOR: 0.7540, F-measure: 0.7124, Precision: 0.7258, Recall: 0.6995, Edit Distance: 0.2042
Image: en_32.png, BLEU: 0.5849, METEOR: 0.7839, F-measure: 0.7921, Precision: 0.8128, Recall: 0.7725, Edit Distance: 0.2004
Image: en_33.png, BLEU: 0.4567, METEOR: 0.6582, F-measure: 0.5679, Precision: 0.5494, Recall: 0.5876, Edit Distance: 0.2814
Image: en_34.png, BLEU: 0.3314, METEOR: 0.6018, F-measure: 0.5512, Precision: 0.5362, Recall: 0.5670, Edit Distance: 0.3647
Image: en_35.png, BLEU: 0.7993, METEOR: 0.9238, F-measure: 0.8796, Precision: 0.8642, Recall: 0.8956, Edit Distance: 0.0763
Image: en_36.png, BLEU: 0.1936, METEOR: 0.2570, F-measure: 0.4199, Precision: 0.4932, Recall: 0.3656, Edit Distance: 0.5985
Image: en_37.png, BLEU: 0.2336, METEOR: 0.4991, F-measure: 0.4488, Precision: 0.4560, Recall: 0.4417, Edit Distance: 0.4372
Image: en_38.png, BLEU: 0.3966, METEOR: 0.7017, F-measure: 0.5825, Precision: 0.5585, Recall: 0.6087, Edit Distance: 0.3202
Image: en_39.png, BLEU: 0.8671, METEOR: 0.9472, F-measure: 0.9323, Precision: 0.9254, Recall: 0.9394, Edit Distance: 0.0589
Image: en_40.png, BLEU: 0.6599, METEOR: 0.8622, F-measure: 0.8064, Precision: 0.7958, Recall: 0.8172, Edit Distance: 0.1326
Image: en_41.png, BLEU: 0.5059, METEOR: 0.5456, F-measure: 0.7284, Precision: 0.7589, Recall: 0.7002, Edit Distance: 0.2466
Image: en_42.png, BLEU: 0.3749, METEOR: 0.6988, F-measure: 0.6156, Precision: 0.5981, Recall: 0.6342, Edit Distance: 0.2671
Image: en_43.png, BLEU: 0.3351, METEOR: 0.6601, F-measure: 0.5416, Precision: 0.5288, Recall: 0.5549, Edit Distance: 0.3410
Image: en_44.png, BLEU: 0.7595, METEOR: 0.9073, F-measure: 0.8750, Precision: 0.8612, Recall: 0.8893, Edit Distance: 0.0676
Image: en_45.png, BLEU: 0.6360, METEOR: 0.8270, F-measure: 0.7971, Precision: 0.7913, Recall: 0.8029, Edit Distance: 0.1642
Image: en_46.png, BLEU: 0.4480, METEOR: 0.7551, F-measure: 0.6892, Precision: 0.6869, Recall: 0.6915, Edit Distance: 0.2486
Image: en_47.png, BLEU: 0.3475, METEOR: 0.6801, F-measure: 0.5776, Precision: 0.6000, Recall: 0.5568, Edit Distance: 0.3049
Image: en_48.png, BLEU: 0.7154, METEOR: 0.8726, F-measure: 0.8110, Precision: 0.7984, Recall: 0.8240, Edit Distance: 0.1141
Image: en_49.png, BLEU: 0.4656, METEOR: 0.7620, F-measure: 0.6772, Precision: 0.6690, Recall: 0.6857, Edit Distance: 0.2318
Image: en_50.png, BLEU: 0.3590, METEOR: 0.6802, F-measure: 0.5780, Precision: 0.5848, Recall: 0.5714, Edit Distance: 0.3471
Image: en_51.png, BLEU: 0.3375, METEOR: 0.6412, F-measure: 0.5823, Precision: 0.5897, Recall: 0.5750, Edit Distance: 0.3450
Image: en_52.png, BLEU: 0.3611, METEOR: 0.6904, F-measure: 0.6151, Precision: 0.6059, Recall: 0.6245, Edit Distance: 0.3345
Image: en_53.png, BLEU: 0.3681, METEOR: 0.6433, F-measure: 0.6092, Precision: 0.6108, Recall: 0.6075, Edit Distance: 0.1751
Image: en_54.png, BLEU: 0.6635, METEOR: 0.8513, F-measure: 0.8031, Precision: 0.7970, Recall: 0.8093, Edit Distance: 0.1566
Image: en_55.png, BLEU: 0.5399, METEOR: 0.7664, F-measure: 0.6918, Precision: 0.6850, Recall: 0.6988, Edit Distance: 0.2713
Image: en_56.png, BLEU: 0.5311, METEOR: 0.7989, F-measure: 0.7474, Precision: 0.7347, Recall: 0.7606, Edit Distance: 0.1984
Image: en_57.png, BLEU: 0.6288, METEOR: 0.8011, F-measure: 0.7818, Precision: 0.8048, Recall: 0.7601, Edit Distance: 0.1661
Image: en_58.png, BLEU: 0.0558, METEOR: 0.3730, F-measure: 0.3150, Precision: 0.3019, Recall: 0.3292, Edit Distance: 0.5139
Image: en_59.png, BLEU: 0.4421, METEOR: 0.7494, F-measure: 0.6559, Precision: 0.6585, Recall: 0.6532, Edit Distance: 0.2176
Image: en_60.png, BLEU: 0.5203, METEOR: 0.7592, F-measure: 0.6883, Precision: 0.6799, Recall: 0.6969, Edit Distance: 0.1944
Image: en_61.png, BLEU: 0.6261, METEOR: 0.8381, F-measure: 0.7782, Precision: 0.7879, Recall: 0.7688, Edit Distance: 0.1609
Image: en_62.png, BLEU: 0.4249, METEOR: 0.7268, F-measure: 0.6264, Precision: 0.6173, Recall: 0.6357, Edit Distance: 0.2625
Image: en_63.png, BLEU: 0.4896, METEOR: 0.7692, F-measure: 0.7447, Precision: 0.7420, Recall: 0.7473, Edit Distance: 0.1786
Image: en_64.png, BLEU: 0.2291, METEOR: 0.5281, F-measure: 0.3604, Precision: 0.3541, Recall: 0.3669, Edit Distance: 0.4516
Image: en_65.png, BLEU: 0.7592, METEOR: 0.8910, F-measure: 0.8136, Precision: 0.8255, Recall: 0.8021, Edit Distance: 0.0783
Image: en_66.png, BLEU: 0.3988, METEOR: 0.4077, F-measure: 0.6770, Precision: 0.7466, Recall: 0.6193, Edit Distance: 0.4673
Image: en_67.png, BLEU: 0.4964, METEOR: 0.7773, F-measure: 0.6519, Precision: 0.6319, Recall: 0.6732, Edit Distance: 0.1564
Image: en_68.png, BLEU: 0.3954, METEOR: 0.7193, F-measure: 0.6197, Precision: 0.6119, Recall: 0.6276, Edit Distance: 0.2635
Image: en_69.png, BLEU: 0.1545, METEOR: 0.4472, F-measure: 0.3505, Precision: 0.3458, Recall: 0.3554, Edit Distance: 0.4463
Image: en_70.png, BLEU: 0.4209, METEOR: 0.6765, F-measure: 0.5920, Precision: 0.5920, Recall: 0.5920, Edit Distance: 0.1888
Image: en_71.png, BLEU: 0.2954, METEOR: 0.6641, F-measure: 0.5297, Precision: 0.5256, Recall: 0.5339, Edit Distance: 0.2225
Image: en_72.png, BLEU: 0.3021, METEOR: 0.6436, F-measure: 0.5611, Precision: 0.5561, Recall: 0.5662, Edit Distance: 0.1943
Image: en_73.png, BLEU: 0.1769, METEOR: 0.4850, F-measure: 0.4425, Precision: 0.4355, Recall: 0.4498, Edit Distance: 0.3569
Image: en_74.png, BLEU: 0.4625, METEOR: 0.7825, F-measure: 0.6912, Precision: 0.6832, Recall: 0.6993, Edit Distance: 0.1922
Image: en_75.png, BLEU: 0.1546, METEOR: 0.4652, F-measure: 0.3941, Precision: 0.3859, Recall: 0.4027, Edit Distance: 0.4560
Image: en_76.png, BLEU: 0.5050, METEOR: 0.7913, F-measure: 0.6972, Precision: 0.6838, Recall: 0.7111, Edit Distance: 0.2460
Image: en_77.png, BLEU: 0.6200, METEOR: 0.8447, F-measure: 0.7859, Precision: 0.7788, Recall: 0.7932, Edit Distance: 0.1936
Image: en_78.png, BLEU: 0.3302, METEOR: 0.6793, F-measure: 0.5597, Precision: 0.5528, Recall: 0.5667, Edit Distance: 0.2580
Image: en_79.png, BLEU: 0.6805, METEOR: 0.7833, F-measure: 0.8351, Precision: 0.9305, Recall: 0.7574, Edit Distance: 0.2657
Image: en_80.png, BLEU: 0.6617, METEOR: 0.8537, F-measure: 0.7650, Precision: 0.7534, Recall: 0.7770, Edit Distance: 0.1052
Image: en_81.png, BLEU: 0.2418, METEOR: 0.5641, F-measure: 0.4806, Precision: 0.4891, Recall: 0.4724, Edit Distance: 0.3625
Image: en_82.png, BLEU: 0.5442, METEOR: 0.8057, F-measure: 0.7356, Precision: 0.7115, Recall: 0.7613, Edit Distance: 0.1770
Image: en_83.png, BLEU: 0.2108, METEOR: 0.4991, F-measure: 0.4521, Precision: 0.4612, Recall: 0.4434, Edit Distance: 0.4400
Image: en_84.png, BLEU: 0.5302, METEOR: 0.7661, F-measure: 0.7434, Precision: 0.7419, Recall: 0.7449, Edit Distance: 0.1877
Image: en_85.png, BLEU: 0.5750, METEOR: 0.8050, F-measure: 0.7882, Precision: 0.7839, Recall: 0.7926, Edit Distance: 0.1644
Image: en_86.png, BLEU: 0.5703, METEOR: 0.8119, F-measure: 0.7391, Precision: 0.7278, Recall: 0.7507, Edit Distance: 0.1549
Image: en_87.png, BLEU: 0.7224, METEOR: 0.8198, F-measure: 0.8346, Precision: 0.8740, Recall: 0.7986, Edit Distance: 0.1554
Image: en_88.png, BLEU: 0.0640, METEOR: 0.2830, F-measure: 0.2510, Precision: 0.2363, Recall: 0.2676, Edit Distance: 0.5803
Image: en_89.png, BLEU: 0.7309, METEOR: 0.8908, F-measure: 0.8308, Precision: 0.8220, Recall: 0.8398, Edit Distance: 0.1234
Image: en_90.png, BLEU: 0.3412, METEOR: 0.6819, F-measure: 0.5868, Precision: 0.5772, Recall: 0.5966, Edit Distance: 0.2092
Image: en_91.png, BLEU: 0.6842, METEOR: 0.8705, F-measure: 0.8021, Precision: 0.7927, Recall: 0.8117, Edit Distance: 0.1070
Image: en_92.png, BLEU: 0.7434, METEOR: 0.8952, F-measure: 0.8421, Precision: 0.8306, Recall: 0.8539, Edit Distance: 0.0727
Image: en_93.png, BLEU: 0.2062, METEOR: 0.5276, F-measure: 0.3819, Precision: 0.3577, Recall: 0.4097, Edit Distance: 0.4581
Image: en_94.png, BLEU: 0.6327, METEOR: 0.8417, F-measure: 0.8468, Precision: 0.8577, Recall: 0.8363, Edit Distance: 0.1335
Image: en_95.png, BLEU: 0.4090, METEOR: 0.7273, F-measure: 0.6206, Precision: 0.6108, Recall: 0.6307, Edit Distance: 0.2655
Image: en_96.png, BLEU: 0.2327, METEOR: 0.5652, F-measure: 0.4501, Precision: 0.4259, Recall: 0.4773, Edit Distance: 0.4196
Image: en_97.png, BLEU: 0.3535, METEOR: 0.6541, F-measure: 0.5294, Precision: 0.5192, Recall: 0.5400, Edit Distance: 0.2509
Image: en_98.png, BLEU: 0.3416, METEOR: 0.6664, F-measure: 0.6012, Precision: 0.5843, Recall: 0.6190, Edit Distance: 0.3357
Image: en_99.png, BLEU: 0.3600, METEOR: 0.6817, F-measure: 0.5911, Precision: 0.5911, Recall: 0.5911, Edit Distance: 0.2976
Image: en_100.png, BLEU: 0.4554, METEOR: 0.7459, F-measure: 0.7135, Precision: 0.7337, Recall: 0.6944, Edit Distance: 0.2557
Image: en_101.png, BLEU: 0.6994, METEOR: 0.8804, F-measure: 0.8327, Precision: 0.8130, Recall: 0.8534, Edit Distance: 0.0905
Image: en_102.png, BLEU: 0.5745, METEOR: 0.8144, F-measure: 0.7450, Precision: 0.7199, Recall: 0.7719, Edit Distance: 0.1522
Image: en_103.png, BLEU: 0.2260, METEOR: 0.5137, F-measure: 0.4068, Precision: 0.3976, Recall: 0.4164, Edit Distance: 0.4058
Image: en_104.png, BLEU: 0.6695, METEOR: 0.8305, F-measure: 0.8044, Precision: 0.7859, Recall: 0.8239, Edit Distance: 0.1090
Image: en_105.png, BLEU: 0.4190, METEOR: 0.7113, F-measure: 0.6801, Precision: 0.7152, Recall: 0.6484, Edit Distance: 0.2734
Image: en_106.png, BLEU: 0.4106, METEOR: 0.7370, F-measure: 0.6714, Precision: 0.6862, Recall: 0.6573, Edit Distance: 0.2786
Image: en_107.png, BLEU: 0.6395, METEOR: 0.8441, F-measure: 0.8327, Precision: 0.8357, Recall: 0.8297, Edit Distance: 0.1520
Image: en_108.png, BLEU: 0.5829, METEOR: 0.8356, F-measure: 0.7186, Precision: 0.7135, Recall: 0.7238, Edit Distance: 0.1749
Image: en_109.png, BLEU: 0.6121, METEOR: 0.8413, F-measure: 0.7517, Precision: 0.7441, Recall: 0.7595, Edit Distance: 0.1223
Image: en_110.png, BLEU: 0.5490, METEOR: 0.7988, F-measure: 0.7210, Precision: 0.7177, Recall: 0.7242, Edit Distance: 0.1681
Image: en_111.png, BLEU: 0.6985, METEOR: 0.8675, F-measure: 0.8620, Precision: 0.8567, Recall: 0.8673, Edit Distance: 0.0953
Image: en_112.png, BLEU: 0.4463, METEOR: 0.7258, F-measure: 0.6737, Precision: 0.6684, Recall: 0.6791, Edit Distance: 0.3268
# 平均指标：
============================================================
Evaluation Results:
============================================================
eval question num: 112
bleu: 0.4552
meteor: 0.7023
f_measure: 0.6358
precision: 0.6341
recall: 0.6376
edit_distance: 0.2744
============================================================

In [ ]:
# en_page_ocr_from_text_with_raw_ocr.json 评估结果：
Image: en_1.png, BLEU: 0.9739, METEOR: 0.9878, F-measure: 0.9728, Precision: 0.9712, Recall: 0.9744, Edit Distance: 0.0173
Image: en_2.png, BLEU: 0.9284, METEOR: 0.9732, F-measure: 0.9516, Precision: 0.9468, Recall: 0.9564, Edit Distance: 0.0249
Image: en_3.png, BLEU: 0.9305, METEOR: 0.9758, F-measure: 0.9679, Precision: 0.9639, Recall: 0.9720, Edit Distance: 0.0340
Image: en_4.png, BLEU: 0.9188, METEOR: 0.9675, F-measure: 0.9426, Precision: 0.9363, Recall: 0.9489, Edit Distance: 0.0172
Image: en_5.png, BLEU: 0.8490, METEOR: 0.9340, F-measure: 0.8993, Precision: 0.8933, Recall: 0.9054, Edit Distance: 0.0364
Image: en_6.png, BLEU: 0.9481, METEOR: 0.9829, F-measure: 0.9713, Precision: 0.9644, Recall: 0.9783, Edit Distance: 0.0191
Image: en_7.png, BLEU: 0.9934, METEOR: 0.9979, F-measure: 0.9955, Precision: 0.9955, Recall: 0.9955, Edit Distance: 0.0118
Image: en_8.png, BLEU: 0.9429, METEOR: 0.9776, F-measure: 0.9638, Precision: 0.9638, Recall: 0.9638, Edit Distance: 0.0266
Image: en_9.png, BLEU: 0.9868, METEOR: 0.9947, F-measure: 0.9923, Precision: 0.9923, Recall: 0.9923, Edit Distance: 0.0296
Image: en_10.png, BLEU: 0.8887, METEOR: 0.9542, F-measure: 0.9292, Precision: 0.9235, Recall: 0.9350, Edit Distance: 0.0299
Image: en_11.png, BLEU: 0.9828, METEOR: 0.9932, F-measure: 0.9837, Precision: 0.9837, Recall: 0.9837, Edit Distance: 0.0247
Image: en_12.png, BLEU: 0.9730, METEOR: 0.9898, F-measure: 0.9872, Precision: 0.9855, Recall: 0.9891, Edit Distance: 0.0264
Image: en_13.png, BLEU: 0.8747, METEOR: 0.9562, F-measure: 0.9144, Precision: 0.8970, Recall: 0.9324, Edit Distance: 0.0298
Image: en_14.png, BLEU: 0.9586, METEOR: 0.9845, F-measure: 0.9616, Precision: 0.9595, Recall: 0.9638, Edit Distance: 0.0389
Image: en_15.png, BLEU: 0.9561, METEOR: 0.9838, F-measure: 0.9717, Precision: 0.9674, Recall: 0.9760, Edit Distance: 0.0253
Image: en_16.png, BLEU: 0.9314, METEOR: 0.9645, F-measure: 0.9540, Precision: 0.9500, Recall: 0.9580, Edit Distance: 0.0307
Image: en_17.png, BLEU: 0.9465, METEOR: 0.9825, F-measure: 0.9553, Precision: 0.9462, Recall: 0.9645, Edit Distance: 0.0346
Image: en_18.png, BLEU: 0.9509, METEOR: 0.9827, F-measure: 0.9790, Precision: 0.9760, Recall: 0.9819, Edit Distance: 0.0364
Image: en_19.png, BLEU: 0.7778, METEOR: 0.9059, F-measure: 0.8871, Precision: 0.8774, Recall: 0.8971, Edit Distance: 0.0528
Image: en_20.png, BLEU: 0.9585, METEOR: 0.9872, F-measure: 0.9745, Precision: 0.9663, Recall: 0.9829, Edit Distance: 0.0205
Image: en_21.png, BLEU: 1.0000, METEOR: 1.0000, F-measure: 1.0000, Precision: 1.0000, Recall: 1.0000, Edit Distance: 0.0142
Image: en_22.png, BLEU: 0.9282, METEOR: 0.9724, F-measure: 0.9542, Precision: 0.9511, Recall: 0.9574, Edit Distance: 0.0249
Image: en_23.png, BLEU: 0.7946, METEOR: 0.8925, F-measure: 0.8359, Precision: 0.8199, Recall: 0.8526, Edit Distance: 0.0710
Image: en_24.png, BLEU: 0.9664, METEOR: 0.9866, F-measure: 0.9766, Precision: 0.9753, Recall: 0.9780, Edit Distance: 0.0138
Image: en_25.png, BLEU: 0.9574, METEOR: 0.9855, F-measure: 0.9809, Precision: 0.9722, Recall: 0.9897, Edit Distance: 0.0142
Image: en_26.png, BLEU: 0.9573, METEOR: 0.9848, F-measure: 0.9699, Precision: 0.9635, Recall: 0.9764, Edit Distance: 0.0180
Image: en_27.png, BLEU: 0.9920, METEOR: 0.9968, F-measure: 0.9951, Precision: 0.9968, Recall: 0.9935, Edit Distance: 0.0133
Image: en_28.png, BLEU: 0.9756, METEOR: 0.9911, F-measure: 0.9775, Precision: 0.9724, Recall: 0.9826, Edit Distance: 0.0311
Image: en_29.png, BLEU: 0.7872, METEOR: 0.9067, F-measure: 0.8372, Precision: 0.8182, Recall: 0.8571, Edit Distance: 0.0483
Image: en_30.png, BLEU: 0.9220, METEOR: 0.9687, F-measure: 0.9600, Precision: 0.9600, Recall: 0.9600, Edit Distance: 0.0315
Image: en_31.png, BLEU: 0.9476, METEOR: 0.9796, F-measure: 0.9779, Precision: 0.9817, Recall: 0.9741, Edit Distance: 0.0338
Image: en_32.png, BLEU: 0.9031, METEOR: 0.9663, F-measure: 0.9606, Precision: 0.9596, Recall: 0.9617, Edit Distance: 0.0229
Image: en_33.png, BLEU: 0.8160, METEOR: 0.9333, F-measure: 0.8959, Precision: 0.8729, Recall: 0.9201, Edit Distance: 0.0272
Image: en_34.png, BLEU: 0.9801, METEOR: 0.9980, F-measure: 0.9904, Precision: 0.9923, Recall: 0.9885, Edit Distance: 0.0220
Image: en_35.png, BLEU: 0.9413, METEOR: 0.9799, F-measure: 0.9710, Precision: 0.9663, Recall: 0.9757, Edit Distance: 0.0147
Image: en_36.png, BLEU: 0.6476, METEOR: 0.5586, F-measure: 0.8556, Precision: 0.9529, Recall: 0.7763, Edit Distance: 0.2701
Image: en_37.png, BLEU: 0.9290, METEOR: 0.9644, F-measure: 0.9409, Precision: 0.9409, Recall: 0.9409, Edit Distance: 0.0312
Image: en_38.png, BLEU: 0.9455, METEOR: 0.9791, F-measure: 0.9566, Precision: 0.9539, Recall: 0.9594, Edit Distance: 0.0180
Image: en_39.png, BLEU: 0.9902, METEOR: 0.9981, F-measure: 0.9924, Precision: 0.9949, Recall: 0.9899, Edit Distance: 0.0235
Image: en_40.png, BLEU: 0.9218, METEOR: 0.9721, F-measure: 0.9653, Precision: 0.9577, Recall: 0.9731, Edit Distance: 0.0222
Image: en_41.png, BLEU: 0.7948, METEOR: 0.6930, F-measure: 0.9124, Precision: 0.9494, Recall: 0.8782, Edit Distance: 0.1298
Image: en_42.png, BLEU: 0.9414, METEOR: 0.9791, F-measure: 0.9568, Precision: 0.9474, Recall: 0.9664, Edit Distance: 0.0201
Image: en_43.png, BLEU: 0.9315, METEOR: 0.9766, F-measure: 0.9590, Precision: 0.9538, Recall: 0.9643, Edit Distance: 0.0259
Image: en_44.png, BLEU: 0.9069, METEOR: 0.9703, F-measure: 0.9547, Precision: 0.9486, Recall: 0.9609, Edit Distance: 0.0229
Image: en_45.png, BLEU: 0.8913, METEOR: 0.9480, F-measure: 0.9195, Precision: 0.9155, Recall: 0.9235, Edit Distance: 0.0253
Image: en_46.png, BLEU: 0.9682, METEOR: 0.9878, F-measure: 0.9780, Precision: 0.9764, Recall: 0.9797, Edit Distance: 0.0243
Image: en_47.png, BLEU: 0.9360, METEOR: 0.9722, F-measure: 0.9564, Precision: 0.9582, Recall: 0.9545, Edit Distance: 0.0414
Image: en_48.png, BLEU: 0.9404, METEOR: 0.9782, F-measure: 0.9590, Precision: 0.9503, Recall: 0.9680, Edit Distance: 0.0232
Image: en_49.png, BLEU: 0.9402, METEOR: 0.9759, F-measure: 0.9590, Precision: 0.9573, Recall: 0.9607, Edit Distance: 0.0253
Image: en_50.png, BLEU: 0.9883, METEOR: 0.9935, F-measure: 0.9914, Precision: 0.9914, Recall: 0.9914, Edit Distance: 0.0156
Image: en_51.png, BLEU: 0.8955, METEOR: 0.9556, F-measure: 0.9276, Precision: 0.9302, Recall: 0.9250, Edit Distance: 0.0344
Image: en_52.png, BLEU: 0.9626, METEOR: 0.9856, F-measure: 0.9652, Precision: 0.9610, Recall: 0.9694, Edit Distance: 0.0378
Image: en_53.png, BLEU: 0.9539, METEOR: 0.9837, F-measure: 0.9723, Precision: 0.9697, Recall: 0.9749, Edit Distance: 0.0177
Image: en_54.png, BLEU: 0.9575, METEOR: 0.9850, F-measure: 0.9782, Precision: 0.9744, Recall: 0.9820, Edit Distance: 0.0142
Image: en_55.png, BLEU: 0.9160, METEOR: 0.9684, F-measure: 0.9658, Precision: 0.9677, Recall: 0.9639, Edit Distance: 0.0214
Image: en_56.png, BLEU: 0.9671, METEOR: 0.9869, F-measure: 0.9772, Precision: 0.9754, Recall: 0.9789, Edit Distance: 0.0234
Image: en_57.png, BLEU: 0.9437, METEOR: 0.9753, F-measure: 0.9646, Precision: 0.9670, Recall: 0.9621, Edit Distance: 0.0322
Image: en_58.png, BLEU: 0.9127, METEOR: 0.9677, F-measure: 0.9547, Precision: 0.9547, Recall: 0.9547, Edit Distance: 0.0397
Image: en_59.png, BLEU: 0.9523, METEOR: 0.9810, F-measure: 0.9658, Precision: 0.9639, Recall: 0.9677, Edit Distance: 0.0365
Image: en_60.png, BLEU: 0.9555, METEOR: 0.9795, F-measure: 0.9641, Precision: 0.9626, Recall: 0.9656, Edit Distance: 0.0211
Image: en_61.png, BLEU: 0.9471, METEOR: 0.9772, F-measure: 0.9718, Precision: 0.9705, Recall: 0.9731, Edit Distance: 0.0160
Image: en_62.png, BLEU: 0.8462, METEOR: 0.9340, F-measure: 0.8930, Precision: 0.8864, Recall: 0.8996, Edit Distance: 0.0338
Image: en_63.png, BLEU: 0.8906, METEOR: 0.9579, F-measure: 0.9502, Precision: 0.9502, Recall: 0.9502, Edit Distance: 0.0217
Image: en_64.png, BLEU: 0.9910, METEOR: 0.9964, F-measure: 0.9899, Precision: 0.9880, Recall: 0.9919, Edit Distance: 0.0201
Image: en_65.png, BLEU: 0.9055, METEOR: 0.9662, F-measure: 0.9298, Precision: 0.9233, Recall: 0.9364, Edit Distance: 0.0179
Image: en_66.png, BLEU: 0.4137, METEOR: 0.7026, F-measure: 0.6444, Precision: 0.6304, Recall: 0.6591, Edit Distance: 0.1317
Image: en_67.png, BLEU: 0.9465, METEOR: 0.9816, F-measure: 0.9579, Precision: 0.9487, Recall: 0.9673, Edit Distance: 0.0341
Image: en_68.png, BLEU: 0.9482, METEOR: 0.9809, F-measure: 0.9813, Precision: 0.9790, Recall: 0.9836, Edit Distance: 0.0132
Image: en_69.png, BLEU: 0.9547, METEOR: 0.9847, F-measure: 0.9705, Precision: 0.9655, Recall: 0.9756, Edit Distance: 0.0221
Image: en_70.png, BLEU: 0.9262, METEOR: 0.9684, F-measure: 0.9509, Precision: 0.9509, Recall: 0.9509, Edit Distance: 0.0234
Image: en_71.png, BLEU: 0.9582, METEOR: 0.9848, F-measure: 0.9741, Precision: 0.9708, Recall: 0.9774, Edit Distance: 0.0167
Image: en_72.png, BLEU: 0.9477, METEOR: 0.9802, F-measure: 0.9663, Precision: 0.9638, Recall: 0.9688, Edit Distance: 0.0173
Image: en_73.png, BLEU: 0.8341, METEOR: 0.9361, F-measure: 0.8882, Precision: 0.8669, Recall: 0.9105, Edit Distance: 0.0262
Image: en_74.png, BLEU: 0.9694, METEOR: 0.9883, F-measure: 0.9797, Precision: 0.9797, Recall: 0.9797, Edit Distance: 0.0160
Image: en_75.png, BLEU: 0.9606, METEOR: 0.9874, F-measure: 0.9717, Precision: 0.9637, Recall: 0.9799, Edit Distance: 0.0169
Image: en_76.png, BLEU: 1.0000, METEOR: 1.0000, F-measure: 1.0000, Precision: 1.0000, Recall: 1.0000, Edit Distance: 0.0434
Image: en_77.png, BLEU: 0.9957, METEOR: 0.9983, F-measure: 0.9969, Precision: 0.9969, Recall: 0.9969, Edit Distance: 0.0228
Image: en_78.png, BLEU: 0.9319, METEOR: 0.9690, F-measure: 0.9459, Precision: 0.9446, Recall: 0.9472, Edit Distance: 0.0185
Image: en_79.png, BLEU: 0.9221, METEOR: 0.9697, F-measure: 0.9332, Precision: 0.9257, Recall: 0.9407, Edit Distance: 0.0489
Image: en_80.png, BLEU: 0.9149, METEOR: 0.9685, F-measure: 0.9379, Precision: 0.9283, Recall: 0.9477, Edit Distance: 0.0195
Image: en_81.png, BLEU: 0.9498, METEOR: 0.9119, F-measure: 0.9800, Precision: 0.9946, Recall: 0.9659, Edit Distance: 0.0484
Image: en_82.png, BLEU: 0.9592, METEOR: 0.9846, F-measure: 0.9572, Precision: 0.9476, Recall: 0.9671, Edit Distance: 0.0262
Image: en_83.png, BLEU: 0.9103, METEOR: 0.9438, F-measure: 0.9412, Precision: 0.9378, Recall: 0.9446, Edit Distance: 0.0278
Image: en_84.png, BLEU: 0.8824, METEOR: 0.9442, F-measure: 0.9032, Precision: 0.8996, Recall: 0.9069, Edit Distance: 0.0254
Image: en_85.png, BLEU: 0.8221, METEOR: 0.9292, F-measure: 0.9483, Precision: 0.9449, Recall: 0.9519, Edit Distance: 0.0388
Image: en_86.png, BLEU: 0.9105, METEOR: 0.9650, F-measure: 0.9529, Precision: 0.9489, Recall: 0.9570, Edit Distance: 0.0360
Image: en_87.png, BLEU: 0.7708, METEOR: 0.8338, F-measure: 0.8515, Precision: 0.8982, Recall: 0.8094, Edit Distance: 0.1445
Image: en_88.png, BLEU: 0.8974, METEOR: 0.9616, F-measure: 0.9305, Precision: 0.9206, Recall: 0.9405, Edit Distance: 0.0215
Image: en_89.png, BLEU: 0.9585, METEOR: 0.9873, F-measure: 0.9636, Precision: 0.9534, Recall: 0.9740, Edit Distance: 0.0261
Image: en_90.png, BLEU: 0.9348, METEOR: 0.9690, F-measure: 0.9648, Precision: 0.9689, Recall: 0.9608, Edit Distance: 0.0156
Image: en_91.png, BLEU: 0.9549, METEOR: 0.9829, F-measure: 0.9709, Precision: 0.9683, Recall: 0.9735, Edit Distance: 0.0139
Image: en_92.png, BLEU: 0.9146, METEOR: 0.9643, F-measure: 0.9371, Precision: 0.9331, Recall: 0.9410, Edit Distance: 0.0171
Image: en_93.png, BLEU: 0.9899, METEOR: 0.9959, F-measure: 0.9912, Precision: 0.9912, Recall: 0.9912, Edit Distance: 0.0226
Image: en_94.png, BLEU: 0.9540, METEOR: 0.9830, F-measure: 0.9698, Precision: 0.9681, Recall: 0.9715, Edit Distance: 0.0201
Image: en_95.png, BLEU: 0.9010, METEOR: 0.9585, F-measure: 0.9477, Precision: 0.9477, Recall: 0.9477, Edit Distance: 0.0261
Image: en_96.png, BLEU: 0.9561, METEOR: 0.9837, F-measure: 0.9683, Precision: 0.9669, Recall: 0.9698, Edit Distance: 0.0176
Image: en_97.png, BLEU: 0.8622, METEOR: 0.9279, F-measure: 0.8767, Precision: 0.8767, Recall: 0.8767, Edit Distance: 0.0346
Image: en_98.png, BLEU: 0.9911, METEOR: 0.9964, F-measure: 0.9980, Precision: 1.0000, Recall: 0.9960, Edit Distance: 0.0297
Image: en_99.png, BLEU: 0.9446, METEOR: 0.9765, F-measure: 0.9696, Precision: 0.9712, Recall: 0.9681, Edit Distance: 0.0329
Image: en_100.png, BLEU: 0.9708, METEOR: 0.9884, F-measure: 0.9839, Precision: 0.9839, Recall: 0.9839, Edit Distance: 0.0191
Image: en_101.png, BLEU: 0.8514, METEOR: 0.9492, F-measure: 0.9144, Precision: 0.8928, Recall: 0.9372, Edit Distance: 0.0233
Image: en_102.png, BLEU: 0.8080, METEOR: 0.9205, F-measure: 0.8603, Precision: 0.8327, Recall: 0.8897, Edit Distance: 0.0458
Image: en_103.png, BLEU: 0.8859, METEOR: 0.9586, F-measure: 0.9421, Precision: 0.9348, Recall: 0.9495, Edit Distance: 0.0222
Image: en_104.png, BLEU: 0.8776, METEOR: 0.9359, F-measure: 0.9218, Precision: 0.9066, Recall: 0.9375, Edit Distance: 0.0492
Image: en_105.png, BLEU: 0.9644, METEOR: 0.9858, F-measure: 0.9973, Precision: 0.9973, Recall: 0.9973, Edit Distance: 0.0137
Image: en_106.png, BLEU: 0.9568, METEOR: 0.9793, F-measure: 0.9747, Precision: 0.9747, Recall: 0.9747, Edit Distance: 0.0396
Image: en_107.png, BLEU: 0.9239, METEOR: 0.9701, F-measure: 0.9378, Precision: 0.9356, Recall: 0.9400, Edit Distance: 0.0381
Image: en_108.png, BLEU: 0.9645, METEOR: 0.9888, F-measure: 0.9741, Precision: 0.9657, Recall: 0.9826, Edit Distance: 0.0216
Image: en_109.png, BLEU: 0.9777, METEOR: 0.9906, F-measure: 0.9828, Precision: 0.9828, Recall: 0.9828, Edit Distance: 0.0163
Image: en_110.png, BLEU: 0.9575, METEOR: 0.9845, F-measure: 0.9713, Precision: 0.9670, Recall: 0.9758, Edit Distance: 0.0329
Image: en_111.png, BLEU: 0.8915, METEOR: 0.9586, F-measure: 0.9309, Precision: 0.9266, Recall: 0.9352, Edit Distance: 0.0206
Image: en_112.png, BLEU: 1.0000, METEOR: 1.0000, F-measure: 1.0000, Precision: 1.0000, Recall: 1.0000, Edit Distance: 0.0419
# 平均指标：
============================================================
Evaluation Results:
============================================================
eval question num: 112
bleu: 0.9282
meteor: 0.9734
f_measure: 0.9553
precision: 0.9521
recall: 0.9586
edit_distance: 0.0284
============================================================